<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/notebooks/multiclass_supervised_unbiased_adaptive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-view Classification Simulation

In [7]:
import numpy as np
from sklearn.datasets import make_blobs
import math
from numba import jit
from scipy.stats import norm
from scipy.optimize import linprog
import os
import sys
import time
import pandas as pd

# To simulate command-line arguments in Colab
class Args:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

## Data Generation and Prediction Helper Functions

In [8]:
def generate_data(nsamples=1000,nclasses=2,nviews=10,seed=42,snrdb=9):
    sval = 10**(snrdb/20)
    rng = np.random.default_rng(seed=seed)
    mm = rng.random(size=(nclasses,nviews)) * sval # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1] #(observations, means, labels)

@jit
def pred_linear_cla(x_observe,class_means):
    mean_tr = class_means.T # (v,nc)
    diff_mean_sq = mean_tr[:,:,None] - mean_tr[:,None,:] # (sel, nc, nc)
    pairwise_mean_avg = 0.5 * (mean_tr[:,:,None] + mean_tr[:,None,:])
    inner_prod = np.sum(
        (x_observe[:,None,None] - pairwise_mean_avg) * diff_mean_sq,
        axis=0) > 0 # bool(nc,nc)
    # mask the diagonal
    np.fill_diagonal(inner_prod,False)
    # prediction
    y_pred = np.argmax(np.sum(inner_prod,axis=1))
    return y_pred

def bool_array_2_int(bool_arr):
    return int("".join(map(str, bool_arr.astype(int))), 2)


def enumerate_subsets_fast(arr):
        arr = arr.astype(int)
        n = len(arr)
        # 1. Convert the array into a binary integer (bitmask)
        parent_mask = 0
        for i in range(n):
            if arr[i] == 1:
                parent_mask |= (1<<i)
        # 2. Enumerate all sub_masks natively
        sub = parent_mask
        out = []
        while True:
            # 3. Convert the integer back to an array
            sub_arr = [0] * n
            for i in range(n):
                if sub & (1<<i):
                    sub_arr[i] = 1
            if np.any(sub_arr) and not np.all(sub_arr==arr):
                out.append(np.array(sub_arr).astype("bool"))
            # Break after processing the empty set (0)
            if sub ==0:
                break

            # The bitwise jump to the next valid subset
            sub = (sub - 1) & parent_mask
        return np.array(out)

## LP Oracle Function

In [9]:
def lp_oracle(reliability,costs_vec, budget_ratio,num_rounds):
    #nviews = len(costs)
    # compute the reward over index
    # solve a lp
    A_eq = np.ones((1,len(reliability)))
    b_eq = np.array([1.0])
    bounds = [(0,None) for _ in range(len(reliability))]
    res = linprog(-reliability,
                  A_ub=costs_vec[None,:],b_ub=budget_ratio,
                  A_eq=A_eq,b_eq=b_eq,
                  bounds=bounds)
    opt_prob = res.x
    opt_reward = -res.fun * num_rounds
    opt_cost = np.sum(opt_prob * costs_vec) * num_rounds
    #print(f"[DEBUG] opt oracle --- reward:{opt_reward:.4f}, cost:{opt_cost:.4f}")
    return {"opt_prob":opt_prob,"opt_reward":opt_reward,"opt_cost":opt_cost}

## Simulation Core Functions

In [10]:
def reliability_learn(means,costs,x_data,y_labels):
    #
    nviews = x_data.shape[1] # features num is view
    diff = np.expand_dims(x_data,axis=1) - means[None,:,:] # (nsamp, class, nviews)
    diff_sq = np.square(diff)
    # now, for all combinations
    rel_est = np.zeros(2**nviews - 1)
    cost_vec = np.zeros(2**nviews - 1)
    for b in range(1,2**nviews):
        b_offset = b -1
        sel_idx = np.array([int(bit) for bit in f"{b:0{nviews}b}"]).astype(bool) # NOTE: make it index selector
        # (make sure 3 dim even single view selection)
        subset_diff = np.sum(diff_sq[:,:,sel_idx],axis=2)
        # output shape (nsamp, class)
        tmp_pred = np.argmin(subset_diff,axis=1)
        rel_est[b_offset] = np.mean(tmp_pred == y_labels)
        cost_vec[b_offset] = np.sum(costs[sel_idx])
    return rel_est, cost_vec

def sim_unbiased(xdata,costs,num_rounds,budget_ratio,means,y_labels,seed,**kwargs):
    rng = np.random.default_rng(seed=seed)
    nclasses = means.shape[0] # (nc, xdim)
    nviews = means.shape[1]
    # init weights
    regular_scale = 1.0
    m_est = rng.normal(size=(nclasses,nviews))
    covs_est = np.stack([regular_scale * np.eye(nviews) for _ in range(nclasses)],axis=0)
    ncnt_mat = np.ones((nclasses,nviews,nviews)) # for stability
    # reliability
    reward_est = np.ones(2**nviews - 1)/nclasses
    reward_cnt = np.ones(2**nviews - 1)
    # subset mappings
    subset_maps_cache = [[] for _ in range(2**nviews - 1)] # on the run
    # compute the cache on the run

    # optimization
    free_indices = np.array([idx for idx in range(nviews) if costs[idx] == 0])

    # init subset: alway include the free index
    alpha_ucb = kwargs.get("alpha_ucb",2.0)
    remain_budget = num_rounds * budget_ratio # initialization

    hedge_v = np.ones(2) # (time, cost)
    hedge_epsilon = np.sqrt(np.log(2)/num_rounds) # by theory
    # recording
    record_acc = np.zeros((num_rounds))
    costs_vec = np.zeros(2**nviews - 1)
    for b in range(1,2**nviews):
        b_offset = b -1
        bsubset = np.array([int(bit) for bit in f"{b:0{nviews}b}"]).astype(bool)
        costs_vec[b_offset] = np.sum(costs[bsubset])
    opt_subsets = kwargs.get("opt_subsets",None)
    opt_prob = kwargs.get("opt_prob",None)

    # pre compute
    costs_mat = np.concatenate([np.ones((1,len(costs_vec))),costs_vec[None,:]],axis=0)

    for t in range(num_rounds):
        if t < 5*nclasses:
            subset = np.ones((nviews)).astype("bool")
        else:
            if opt_subsets and opt_prob:
                # NOTE: subset oracle mode
                subset = rng.choice(opt_subsets,p=opt_prob)
            else:
                # use the ucb for reliability
                bonus_conf = np.sqrt(alpha_ucb*np.log(t+1) / reward_cnt) + reward_est
                hedge_y = hedge_v / np.sum(hedge_v)
                est_cost_vec = np.sum(costs_mat * hedge_y[:,None],axis=0) # (2**nviews -1) vec
                max_bang_per_buck = bonus_conf / est_cost_vec
                max_idx = np.argmax(max_bang_per_buck)
                # NOTE: shift the index offset by 1 due to the no view case
                subset = np.array([int(bit) for bit in f"{max_idx+1:0{nviews}b}"]).astype("bool")

                #sol_dict = lp_oracle(bonus_conf,costs_vec,budget_ratio,num_rounds)
                #sol_prob = np.clip(sol_dict['opt_prob'],min=0,max=1.0)
                #set_idx = rng.choice(np.arange(2**nviews-1),p=sol_prob)
                #subset = np.array([int(bit) for bit in f"{set_idx+1:0{nviews}b}"]).astype(bool)



        # checking the budget
        inst_cost = np.sum(costs[subset])
        # resource vec
        resource_z = np.array([1.0, inst_cost]) # normalized cost

        if remain_budget >= inst_cost:
            remain_budget -= inst_cost
        else:
            break # system stop when budget is exhausted
            # override with free indices
            subset = np.zeros((nviews)).astype("bool")
            subset[free_indices] = True
        # observe
        tmp_cnt = np.diagonal(ncnt_mat,axis1=1,axis2=2) # (nc, nview)
        mean_est =  m_est / tmp_cnt # (nc,nviews)

        x_obs = xdata[t,subset]
        # estimate the statistics based on partial counters and partial sums
        sel_mean = mean_est[:,subset] # (nc, sel)
        y_pred = pred_linear_cla(x_obs,sel_mean)
        # for reward and update (external)
        y_true = y_labels[t]
        reward = y_pred == y_true

        # subset update
        bmask_idx = bool_array_2_int(subset) -1 # offset 1
        # Update the running average for the chosen subset's reliability
        reward_est[bmask_idx] += (reward - reward_est[bmask_idx]) / (reward_cnt[bmask_idx] + 1)
        reward_cnt[bmask_idx] += 1
        # cache check
        if np.sum(subset) == 1:
            subsubsets = []
        elif len(subset_maps_cache[bmask_idx])!=0:
            subsubsets = subset_maps_cache[bmask_idx]
        else:
            # compute once
            subsubsets = enumerate_subsets_fast(subset)
            subset_maps_cache[bmask_idx] = subsubsets

        # go through all subsets and make pseudo prediction again
        for sub in subsubsets:
            # if correct, update the reliability accordingly
            sub_mean = mean_est[:,sub]
            y_sub_pred = pred_linear_cla(xdata[t,sub],sub_mean)
            sb_index = bool_array_2_int(sub) - 1 # offset 1
            # Update the running average for the sub-subset's reliability
            sub_reward = (y_sub_pred == y_true)
            reward_est[sb_index] += ( sub_reward - reward_est[sb_index]) / (reward_cnt[sb_index] + 1)
            reward_cnt[sb_index] += 1

        # update hedge
        hedge_v = hedge_v * ((1+hedge_epsilon) ** (resource_z))

        # update weights
        x_masked = xdata[t] * subset.astype("float")
        m_est[y_true] += x_masked
        covs_est[y_true] += np.outer(x_masked,x_masked)
        ncnt_mat[y_true] += np.outer(subset.astype("int"),subset.astype("int"))

        # monitoring
        # best_combo = np.argmax(reward_est)
        # lcb_best = reward_est[best_combo] - alpha_ucb * np.sqrt(np.log(t+1) / reward_cnt[best_combo])
        # # check how many ucb failed to reach lcb of the best arm
        # for b in range(2**nviews - 1):
        #     tmp_ucb = reward_est[b] + alpha_ucb * np.sqrt(np.log(t+1) / reward_cnt[b])
        #     if tmp_ucb < lcb_best:
        #         print(f"[DEBUG] time {t} -- combo index {b} is potentially a bad arm")

        # record results
        record_acc[t] = reward
    # checking learned mean
    spending = budget_ratio * num_rounds - remain_budget
    reward = np.sum(record_acc)

    return {"reward":reward,"spending":spending,"avg_acc":np.mean(record_acc),"record_acc":record_acc}

## Main Simulation Logic

In [11]:
def main(args):
    # recording
    result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[], "opt_reward":[], "opt_cost":[]}
    # simulation
    time_start = time.perf_counter()

    # fixed the dataset
    rng = np.random.default_rng(seed=args.seed)
    approx_samples = args.num_classes * 2000 # extra samples to reduce reliability estimation error
    x_data_full, means, y_labels_full = generate_data(args.num_rounds + approx_samples,
                                                    args.num_classes,
                                                    args.num_views,
                                                    args.seed,
                                                    snrdb=args.snr)
    cost_vec = rng.random(size=(args.num_views,)) + 0.1 # avoid near zero cost
    # NOTE: free view disabled
    #cost_vec[0] = 0.0 # adding one free view
    cost_vec /= np.sum(cost_vec) # NOTE: this could cause numerical error

    # run the optimal policy once
    approx_reliability, combo_costs_vec = reliability_learn(means,cost_vec,x_data_full,y_labels_full)
    opt_dict = lp_oracle(approx_reliability,combo_costs_vec,args.budget_ratio,args.num_rounds)
    opt_clean = []
    for idx, prob in enumerate(opt_dict['opt_prob']):
        b_offset = idx +1
        if prob > 0:
            #print(f"[DEBUG] optimal subset index:{idx}")
            tmp_subset = np.array([int(bit) for bit in f"{b_offset:0{args.num_views}b}"]).astype("bool")
            #print(f"opt subset:{tmp_subset}")
            opt_clean.append((prob, tmp_subset))
    opt_prob, opt_subsets = zip(*opt_clean)

    x_data = x_data_full[:args.num_rounds]
    y_labels = y_labels_full[:args.num_rounds]
    for t in range(args.num_trials):
        trial_start = time.perf_counter()
        sim_dict = sim_unbiased(
            x_data,
            cost_vec,
            args.num_rounds,
            args.budget_ratio,
            means,
            y_labels,
            args.seed+t,
            **{
                "step_size":args.step_size,
                #"opt_subsets":opt_subsets,
                #"opt_prob":opt_prob,
                })
        print(f"Trial {t + 1}/{args.num_trials}... (Time elapsed: {time.perf_counter() - trial_start:.3f}s)")
        # accumulating
        result_pd['trial'].append(t)
        result_pd['reward'].append(sim_dict['reward'])
        result_pd['spending'].append(sim_dict['spending'])
        result_pd['avg_acc'].append(sim_dict['avg_acc'])
        result_pd['opt_reward'].append(opt_dict['opt_reward'])
        result_pd['opt_cost'].append(opt_dict['opt_cost'])
    res_df = pd.DataFrame.from_dict(result_pd)
    # Save to a temporary CSV file, or just display
    # For Colab, we'll return the DataFrame directly or save to a known path.
    print(f"Simulation complete, total time elapsed: {time.perf_counter() - time_start:.3f}s")
    return res_df

## Define Arguments and Run Simulation

In [12]:
# Define simulation parameters
# These parameters mimic the command-line arguments in the original script.
sim_args = Args(
    num_trials=20,
    num_rounds=1500,
    num_classes=4,
    num_views=10,
    seed=42,
    snr=9,
    budget_ratio=1.0,
    step_size=0.1, # Assuming a default step_size, as it's passed as a kwarg
    output='simulation_results.csv' # Output file name
)

# Run the simulation
results_df = main(sim_args)

# Display the results
display(results_df.head())

Trial 1/20... (Time elapsed: 28.711s)
Trial 2/20... (Time elapsed: 20.433s)
Trial 3/20... (Time elapsed: 23.913s)
Trial 4/20... (Time elapsed: 19.990s)
Trial 5/20... (Time elapsed: 26.382s)
Trial 6/20... (Time elapsed: 16.745s)
Trial 7/20... (Time elapsed: 27.554s)
Trial 8/20... (Time elapsed: 9.754s)
Trial 9/20... (Time elapsed: 8.994s)
Trial 10/20... (Time elapsed: 5.715s)
Trial 11/20... (Time elapsed: 14.722s)
Trial 12/20... (Time elapsed: 11.481s)
Trial 13/20... (Time elapsed: 6.615s)
Trial 14/20... (Time elapsed: 16.537s)
Trial 15/20... (Time elapsed: 6.718s)
Trial 16/20... (Time elapsed: 21.095s)
Trial 17/20... (Time elapsed: 13.336s)
Trial 18/20... (Time elapsed: 17.346s)
Trial 19/20... (Time elapsed: 13.052s)
Trial 20/20... (Time elapsed: 12.371s)
Simulation complete, total time elapsed: 323.194s


,trial,reward,spending,avg_acc,opt_reward,opt_cost
0,0,1288.0,1280.216402,0.858667,1352.368421,1500.0
1,1,1262.0,1198.880802,0.841333,1352.368421,1500.0
2,2,1292.0,1309.494447,0.861333,1352.368421,1500.0
3,3,1286.0,1263.838186,0.857333,1352.368421,1500.0
4,4,1296.0,1338.274702,0.864000,1352.368421,1500.0
